# HiveMind Agentic Benchmark: 4-Harness Arena (Core Suite & Polyglot Python)

Hardware-Optionen: **Tesla T4 (16 GB)**, **L4 (24 GB)** oder **A100 (40/80 GB)**, High-RAM, >=100 GB Disk.

---

## 1. Die drei Benchmark-Ziele

| Ziel | Hardware | Modell / Kontext | Suites & Tasks | Getestete Harnesses | Laufzeit-Erwartung |
|---|---|---|---|---|---|
| **Goal 1: Small-GPU-Baseline** | Tesla T4 (16 GB) / L4 (24 GB) | `qwen3.5:4b-mtp`<br>Context: **128k** (131.072 Tokens) | **23 Core Tasks** + **34 Polyglot Tasks**<br>3 Runs für statistische Varianz | `hivemind_duo`, `hivemind_duo_full`, `aider`, `hermes`, `pi` | 1 Run (2 Harnesses): ~2h<br>1 Run (4 Harnesses): ~5-7h<br>3 Runs (4-5 Harnesses): **8–25h** |
| **Goal 2: Local Hardware Ceiling** | L4 (24 GB) / A100 (40–80 GB) | `qwen3.6:35b-a3b-ud`<br>Context: **32k** oder **64k** | **23 Core Tasks** + **34 Polyglot Tasks** | `hivemind_duo`, `hivemind_duo_full`, `aider`, `hermes`, `pi` | 1 Run (2 Harnesses): ~2.5h<br>1 Run (4 Harnesses): ~6-9h |
| **Goal 3: DeepSWE Real-World** | A100 (40–80 GB) | `qwen3.6:35b-a3b-ud`<br>Context: **128k** (131.072 Tokens) | **3 echte Repo-Tasks** (`tomlkit`, `httpx`, `cattrs`), 1 Run | `hivemind_duo`, `aider`, `hermes`, `pi` | ~1.5–2h (1800s Timeout pro Task) |

> [!NOTE]
> **4B mit 128k auf Tesla T4 (16 GB):** Qwen 3.5 4B MTP nutzt Hybrid-Attention (nur 8 Attention-Layers). Mit `q4_0` KV-Cache belegt der 128k Cache nur ~1.14 GB, Gesamtspeicher ~3.54 GB VRAM. Sollte Colab dennoch ein VRAM-Limit melden, einfach auf `HM_CTX=32768` oder `65536` umschalten.
>
> **DeepSWE:** Es handelt sich um 3 reale Python-Repository-Bugfixes (nicht die vollen 113 Tasks von DeepSWE), damit der Lauf in realistischer Zeit und ohne Docker abgeschlossen werden kann.

---

## 2. Getestete HiveMind Toolsets & Competitor Harnesses

Wir vergleichen nicht Modelle, sondern **Agenten-Architekturen & Harnesses** auf exakt demselben Modell und `llama-server`:

### HiveMind Toolsets:
1. **`hivemind_duo` (Standard: `duo_core`, 18 Tools):**
   - **File & AST:** `read_file`, `get_signatures`, `find_references`, `list_dir`, `find_files`, `search_code`, `lsp_diagnostics`
   - **Edits:** `edit_file`, `write_file`, `write_file_append`, `undo_last`
   - **Execution & Verification:** `run_bash`, `run_python`, `run_tests`
   - **Interaction & Completion:** `task_complete`, `ask_user`, `get_datetime`, `send_file`
2. **`hivemind_duo_full` (Full Toolset):**
   - Alle 18 Core-Tools PLUS AST-Strukturansicht (`view_ast`, `search_ast`), `diff_working_copy`, Web-Scraping (`scrape_page`, `web_search`) und Bildanalyse (`view_image`).

### Competitor Harnesses:
3. **`aider`:** Offizielle Aider CLI (Diff Engine, Repo-Map via tree-sitter).
4. **`hermes`:** Nous Research Hermes Agent CLI (`NousResearch/hermes-agent`).
5. **`pi`:** Mario Zechner Pi Coding Agent CLI (`earendil-works/pi`).

> [!IMPORTANT]
> **Token-Metriken:** Hermes und Pi liefern standardmäßig keine Prompt/Completion-Token zurück (`tokens: 0`). Der präzise Net-Token- und Cache-Vergleich erfolgt daher zwischen `hivemind_duo` und `aider`. Über alle 4/5 Harnesses hinweg werden **Pass-Rate (Resolved %)**, **Wall-Clock Time (Sekunden)**, **AST-Integrität** und **Speichereffizienz** verglichen.

---

## 3. Empfohlene Ausführungs-Reihenfolge (Best Practice)

1. **Schritt 1 (Sanity Check):** Zelle 5 (Smoke Test: 3 Tasks, HiveMind vs. Aider) ~5-10 min.
2. **Schritt 2 (Phase 1 Sanity):** Zelle 6a mit `HM_CONDITIONS='hivemind_duo,aider'`, `HM_RUNS=1` (~30-45 min).
3. **Schritt 3 (Phase 1 Arena):** Zelle 6a mit `HM_CONDITIONS='hivemind_duo,aider,hermes,pi'`, `HM_RUNS=1` (~1.5h).
4. **Schritt 4 (Phase 2 Polyglot):** Zelle 7b mit 1 Run (~2-4h).
5. **Schritt 5 (Phase 3 DeepSWE):** Zelle 8b & 8c nur auf A100 mit 35B / 128k (~1.5h).
6. **Schritt 6 (Varianz-Lauf):** Wenn alle Smoke- und Einzelläufe grün sind, `HM_RUNS=3` für 3 Durchläufe starten.

In [ ]:
# 1) GPU & Disk prüfen
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!df -h /root | tail -1

In [ ]:
# 2) Neuesten bench-stable Branch klonen / aktualisieren
![ -d /root/HiveMind_llama/.git ] || git clone --branch bench-stable https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama
!cd /root/HiveMind_llama && git fetch origin bench-stable && git checkout bench-stable && git reset --hard origin/bench-stable
!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!cd /root/HiveMind_llama && git log -1 --format='%h %s'

In [ ]:
# 3) Basis-Setup (~2-3 min: Python 3.14, llama.cpp, Aider, Hermes & Pi installieren)
# Installiert alle Tools und das Environment ohne langes Modell-Warten
!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!HM_MODEL="skip" HM_LLAMA_CUDA=12.8 bash /root/rent_bootstrap.sh setup

In [ ]:
# 3a) [DOWNLOAD OPTION A] 35B Modell herunterladen (~22 GB, ~10-15 min)
# Qwen 3.6 35B A3B UD-Q4_K_XL (MoE High-Performance Coder)
!bash /root/rent_bootstrap.sh download qwen3.6:35b-a3b-ud

In [ ]:
# 3b) [DOWNLOAD OPTION B] 4B Modell herunterladen (~3 GB, nur ~1-2 min)
# Qwen 3.5 4B MTP (Kompaktes 128k Modell - spart VRAM & Kosten, läuft auf jeder GPU wie T4)
!bash /root/rent_bootstrap.sh download qwen3.5:4b-mtp

In [ ]:
# 4a) [SERVER OPTION 1] 35B Modell in den VRAM laden & starten
# Qwen 3.6 35B MoE | Auto-VRAM Budget (L4: 2 MoE offloaded, A100: 100% VRAM)
import os
os.environ["HM_MODEL"] = "qwen3.6:35b-a3b-ud"
os.environ["HM_CTX"] = "32768"

!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!HM_MODEL="qwen3.6:35b-a3b-ud" HM_CTX=32768 bash /root/rent_bootstrap.sh serve
!curl --max-time 5 -fsS http://127.0.0.1:8001/health
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

In [ ]:
# 4b) [SERVER OPTION 2] 4B Modell mit 128k Kontext in den VRAM laden & starten
# Qwen 3.5 4B MTP | Auto-Download falls fehlt (~2 Min) | T4: 128k machbar (~3.5GB), bei VRAM-Limit HM_CTX=32768
import os
os.environ["HM_MODEL"] = "qwen3.5:4b-mtp"
os.environ["HM_CTX"] = "131072"

!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!HM_MODEL="qwen3.5:4b-mtp" HM_CTX=131072 bash /root/rent_bootstrap.sh serve
!curl --max-time 5 -fsS http://127.0.0.1:8001/health
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

In [ ]:
# 4c) VRAM- & Modell-Status prüfen
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
!echo "Aktives Modell:  $(cat /root/HiveMind_llama/.active_model 2>/dev/null || echo 'nicht gesetzt')"
!echo "Aktiver Kontext: $(cat /root/HiveMind_llama/.active_ctx 2>/dev/null || echo 'nicht gesetzt') Tokens"

In [ ]:
# 5) Smoke-Test (3 Tasks: HiveMind vs. Aider, ~5-10 min)
# Nutzt automatisch das aktive Modell aus Zelle 4a (35B) oder 4b (4B)
!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh smoke

---
## PHASE 1: HiveMind Core Suite (23 Tasks)
Testet Multi-File-Navigation, Refactoring, AST-Symbolerhalt und Selbstkorrektur.
Head-to-Head: HiveMind (`duo_core` oder `duo_full`) vs. `aider`, `hermes`, `pi` auf identischem Modell.

In [ ]:
# 6a) Phase 1 starten: Core Suite (23 Tasks)
# Nutzt automatisch das aktive Modell aus Zelle 4a (35B) oder 4b (4B)
# Wähle Bedingungen:
#   - Schnell / Sanity: 'hivemind_duo,aider' (~30-45 min)
#   - Volle 4-Harness Arena: 'hivemind_duo,aider,hermes,pi' (~1.5-2h)
#   - Core vs. Full Toolset: 'hivemind_duo,hivemind_duo_full,aider,hermes,pi'
import os
os.environ['HM_RUNS'] = '1'  # Starte mit '1' für Sanity Check; setze auf '3' für finale statistische Varianz
os.environ['HM_RUN_OFFSET'] = '0'  # Bei separaten Einzelläufen: Run 1='0', Run 2='1', Run 3='2'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,aider,hermes,pi')
os.environ['HM_TASKS'] = 'bench/tasks'
!bash /root/rent_bootstrap.sh full

In [ ]:
# 6b) Optional: Claude Code auf derselben Core Suite ausführen (braucht ANTHROPIC_API_KEY)
import os
if os.environ.get('ANTHROPIC_API_KEY'):
    !cd /root/HiveMind_llama && .venv/bin/python bench/external/run_external.py --harness claude --tasks bench/tasks
else:
    print('Kein ANTHROPIC_API_KEY gesetzt — Claude Code wird übersprungen.')

In [ ]:
# 6c) Live-Status der Phase 1 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard (aktualisiert alle 10s; Interrupt stoppt nur die Anzeige):
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py

---
## PHASE 2: Polyglot Python Practice Benchmark (34 Tasks)
Das offizielle Benchmark-Set von Aider AI (`Aider-AI/polyglot-benchmark/python/exercises/practice`).
Testet den kompletten 34-Aufgaben-Corpus im direkten Head-to-Head auf demselben llama-server.
Laufzeit: 1 Run mit 2 Harnesses ca. 1.5–2.5h; 1 Run mit 4 Harnesses ca. 4–6h.

In [ ]:
# 7a) Alle 34 Polyglot-Python-Practice-Aufgaben von Aider AI herunterladen und importieren
!HM_POLYGLOT_NAMES=all bash /root/rent_bootstrap.sh polyglot

In [ ]:
# 7b) Phase 2 starten: Polyglot Python Practice (34 Tasks)
# Nutzt automatisch das Modell aus Zelle 4a (35B) oder 4b (4B)
# Wähle Bedingungen:
#   - Schnell / Sanity: 'hivemind_duo,aider' (~1.5-2.5h)
#   - Volle 4-Harness Arena: 'hivemind_duo,aider,hermes,pi' (~4-6h)
#   - Core vs. Full Toolset: 'hivemind_duo,hivemind_duo_full,aider,hermes,pi'
import os
os.environ['HM_RUNS'] = '1'  # Starte mit '1'; setze auf '3' für 3 Durchläufe (Achtung: 3 Runs x 4 Harnesses = 408 Durchläufe, ~12-18h!)
os.environ['HM_RUN_OFFSET'] = '0'  # Bei separaten Einzelläufen: Run 1='0', Run 2='1', Run 3='2'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,aider,hermes,pi')
os.environ['HM_TASKS'] = 'bench/tasks_polyglot'
!bash /root/rent_bootstrap.sh full

In [ ]:
# 7c) Optional: Claude Code auf dem vollen Aider-Benchmark ausführen
import os
if os.environ.get('ANTHROPIC_API_KEY'):
    !cd /root/HiveMind_llama && .venv/bin/python bench/external/run_external.py --harness claude --tasks bench/tasks_polyglot
else:
    print('Kein ANTHROPIC_API_KEY gesetzt — Claude Code wird übersprungen.')

In [ ]:
# 7d) Live-Status der Phase 2 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard:
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py

---
## PHASE 3: DeepSWE Real-World Benchmark (Echte Repositories, 128k Kontext)
Schwere, reale Software-Engineering-Aufgaben aus `datacurve-ai/deep-swe` (`tomlkit`, `httpx`, `cattrs`).
Hier wird die volle Kontextstärke (128k / 131.072 Tokens) über alle Harnesses (`hivemind_duo`, `aider`, `hermes`, `pi`) getestet.
**Hardware-Voraussetzung:** Nur A100 GPU (40/80 GB) + 35B Modell; nicht auf L4/T4 ausführen.
VRAM-Bedarf bei 128k: ~33 GB VRAM (passt perfekt in die Colab A100 mit 40/80 GB).

In [ ]:
# 8a) DeepSWE Python-Tasks importieren (lädt Repos & isolierte Hidden Tests ohne Docker)
# Installiert auch Test-Abhängigkeiten wie hypothesis und trio für den Test-Runner
!pip install -q hypothesis trio attrs pytest pytest-asyncio anyio
!bash /root/rent_bootstrap.sh deepswe

In [ ]:
# 8b) Engine mit 128k Kontext neu starten (~1 min)
# Nur auf A100 (40/80 GB) ausführen!
import os
os.environ["HM_MODEL"] = "qwen3.6:35b-a3b-ud"
os.environ["HM_CTX"] = "131072"

!ln -sf /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh 2>/dev/null || cp -f /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!bash /root/rent_bootstrap.sh stop
!pkill -9 -f llama-server 2>/dev/null || true
!HM_MODEL="qwen3.6:35b-a3b-ud" HM_CTX=131072 bash /root/rent_bootstrap.sh serve
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

In [ ]:
# 8c) Phase 3 starten: DeepSWE Real-World Tasks (128k Kontext)
# 3 echte Repository-Tasks (tomlkit, httpx, cattrs)
import os
os.environ['HM_RUNS'] = '1'  # Bei DeepSWE (1800s Timeout pro Task) empfiehlt sich 1 Durchlauf
os.environ['HM_RUN_OFFSET'] = '0'  # Bei separaten Einzelläufen: Run 1='0', Run 2='1'
os.environ['HM_CONDITIONS'] = os.environ.get('HM_CONDITIONS', 'hivemind_duo,aider,hermes,pi')
os.environ['HM_TASKS'] = 'bench/tasks_deepswe'
os.environ['HM_CTX'] = '131072'
os.environ['HM_CELL_TIMEOUT'] = '1800'
!bash /root/rent_bootstrap.sh full

In [ ]:
# 8d) Live-Status der Phase 3 überwachen (einmalig oder als Live-Monitor)
# Einmaliger Status:
!bash /root/rent_bootstrap.sh status
# ODER Live-Dashboard:
# !cd /root/HiveMind_llama && .venv/bin/python bench/watch.py

---
## AUSWERTUNG & REPORT (Das vereinte Leaderboard)
Analysiert die Ergebnisse wissenschaftlich (Wilson 95% CI, McNemar-Test, Net-Token-Ratio vs. Aider).

In [ ]:
# 9) Unified Leaderboard generieren
# 9a) Getrennt nach Modell & Harness (z. B. hivemind_duo@35b vs. aider@35b vs. hivemind_duo@4b):
!cd /root/HiveMind_llama && .venv/bin/python bench/analyze.py --by-model

# 9b) Gepoolt über alle Runs hinweg (mit Wilson 95% CI und McNemar-Test):
# !cd /root/HiveMind_llama && .venv/bin/python bench/analyze.py

In [ ]:
# 10) Alle JSON-Ergebnisse dauerhaft in Google Drive sichern
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/hivemind_bench
    !cp -v /root/HiveMind_llama/bench/results/*.json /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true
    !cp -v /root/HiveMind_llama/bench/results/external/*.json /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true
    !cp -v /root/HiveMind_llama/bench_full.log /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true
    print('Ergebnisse erfolgreich in Google Drive gesichert (/content/drive/MyDrive/hivemind_bench/)!')
except ImportError:
    print('Nicht in Google Colab (google.colab nicht verfügbar). Ergebnisse liegen in /root/HiveMind_llama/bench/results/')

In [ ]:
# 11) Server stoppen
!bash /root/rent_bootstrap.sh stop